# Pegelonline schematized feeder

[Source overview](https://github.com/clemensv/real-time-sources/tree/main/feeders/pegelonline) | [Schema pilot](../fabric/schematized/README.md)

Runs the existing AMQP feeder with binary JSON CloudEvents against an explicitly selected registered Eventstream source. Eventstream owns the KQL base tables. Attach the source-specific published Fabric Environment (core, AMQP bridge, generated data and AMQP producer wheels) and a Lakehouse before execution. No packages are installed by cells.

Connection credentials are retrieved from the public Topology API at runtime and are never notebook parameters. SCHEMA_BINDINGS pins contract versions and fingerprints; deployment resolves and verifies registered URIs and injects the mapping cell in the deployed copy only. Keep deployment bindings and receipts outside the repository. Dedupe state survives runs. Deploy with ONCE_MODE=True and schedule repeated bounded cycles. RUN_DURATION limits worker execution after the kernel starts.

In [ ]:
EVENTSTREAM_NAME = "pegelonline-ingest"
SOURCE_NAME = "pegelonline-input"
PAYLOAD_ENCODING = "json"
SCHEMA_BINDINGS = {
    "de.wsv.pegelonline.Station": {
        "source_uri": "xreg://./schemagroups/de.wsv.pegelonline.jstruct/schemas/de.wsv.pegelonline.Station/versions/1",
        "projected_version": "1",
        "schema_sha256": "41ca972608fcc2648d5b100a5442d6fe55bbdf6e5b82ab6818c9ec813be403cd"
    },
    "de.wsv.pegelonline.CurrentMeasurement": {
        "source_uri": "xreg://./schemagroups/de.wsv.pegelonline.jstruct/schemas/de.wsv.pegelonline.CurrentMeasurement/versions/1",
        "projected_version": "2",
        "schema_sha256": "74a4d09f63090874a439dbf9b91e71e6be2278ad04d53bc41643eefd2f864b99"
    }
}
WORKSPACE_ID = ""
STATE_FILE = "/lakehouse/default/Files/feeder-state/pegelonline/state-registered.json"
POLLING_INTERVAL = 60
RUN_DURATION = 240
ONCE_MODE = True


In [ ]:
import datetime
import logging
import os
import pathlib
import threading
import traceback
from urllib.parse import urlparse

import notebookutils
import requests

FABRIC_API = 'https://api.fabric.microsoft.com/v1'
LOG_PATH = '/lakehouse/default/Files/feeder-state/pegelonline/last-run.log'

def lookup_es_binding(workspace_id, eventstream_name, source_name):
    headers = {'Authorization': 'Bearer ' + notebookutils.credentials.getToken('pbi')}
    def get(path):
        response = requests.get(FABRIC_API + path, headers=headers, timeout=30)
        response.raise_for_status()
        return response.json()
    items = []
    path = f'/workspaces/{workspace_id}/eventstreams'
    while path:
        page = get(path)
        items.extend(page.get('value', []))
        uri = page.get('continuationUri')
        if uri and not uri.startswith(FABRIC_API + '/'):
            raise RuntimeError('Unexpected Eventstream continuation origin')
        path = uri[len(FABRIC_API):] if uri else None
    matches = [item for item in items if item['displayName'] == eventstream_name]
    if len(matches) != 1:
        raise RuntimeError('Expected exactly one named Eventstream')
    prefix = f'/workspaces/{workspace_id}/eventstreams/{matches[0]["id"]}'
    topology = get(prefix + '/topology')
    sources = [source for source in topology.get('sources', [])
               if source.get('type') == 'CustomEndpoint' and source.get('name') == source_name]
    if len(sources) != 1:
        raise RuntimeError('Expected exactly one named CustomEndpoint source')
    connection = get(prefix + f'/sources/{sources[0]["id"]}/connection')
    return matches[0]['id'], sources[0]['id'], connection['accessKeys']['primaryConnectionString']

def amqp_environment(connection_string):
    parts = dict(part.split('=', 1) for part in connection_string.split(';') if part)
    endpoint = urlparse(parts['Endpoint'])
    if endpoint.scheme != 'sb' or not endpoint.hostname or not parts.get('EntityPath'):
        raise ValueError('Expected an Eventstream Event Hubs source connection')
    return {'AMQP_HOST': endpoint.hostname, 'AMQP_PORT': '5671', 'AMQP_TLS': 'true',
            'AMQP_ADDRESS': parts['EntityPath'], 'AMQP_AUTH_MODE': 'sas',
            'AMQP_SAS_KEY_NAME': parts['SharedAccessKeyName'],
            'AMQP_SAS_KEY': parts['SharedAccessKey'], 'AMQP_CONTENT_MODE': 'binary'}


def lookup_es_connection_string(workspace_id, eventstream_name, source_name):
    return lookup_es_binding(workspace_id, eventstream_name, source_name)[2]


In [ ]:
# Deployment replaces this cell in an external copy after schema fingerprint verification.
dataschema_overrides = None


In [ ]:
pathlib.Path(LOG_PATH).parent.mkdir(parents=True, exist_ok=True)
log_handler = logging.FileHandler(LOG_PATH, mode='w', encoding='utf-8')
log_handler.setFormatter(logging.Formatter('%(asctime)s %(levelname)s %(message)s'))
root_logger = logging.getLogger()
root_logger.addHandler(log_handler)
old_level = root_logger.level
root_logger.setLevel(logging.INFO)
errors = []
class ErrorRecorder(logging.Handler):
    def emit(self, record):
        if record.levelno >= logging.ERROR:
            errors.append(record.getMessage())
error_handler = ErrorRecorder()
root_logger.addHandler(error_handler)
result = 'OK'
try:
    if RUN_DURATION <= 0 or POLLING_INTERVAL <= 0:
        raise ValueError('RUN_DURATION and POLLING_INTERVAL must be positive')
    from proton import SSL
    if not SSL.present():
        raise RuntimeError('Environment Proton wheel was built without TLS support')
    os.environ.setdefault('SSL_CERT_FILE', requests.certs.where())
    workspace_id = WORKSPACE_ID or notebookutils.runtime.context['currentWorkspaceId']
    content_types = {'json': 'application/json', 'avro': 'application/vnd.apache.avro+avro'}
    if PAYLOAD_ENCODING not in content_types:
        raise ValueError('PAYLOAD_ENCODING must be json or avro')
    content_type = content_types[PAYLOAD_ENCODING]
    expected_schema_uris = {binding['source_uri'] for binding in SCHEMA_BINDINGS.values()}
    if not isinstance(dataschema_overrides, dict) or set(dataschema_overrides) != expected_schema_uris:
        raise RuntimeError('Deploy the notebook with a verified schema mapping cell before running')
    eventstream_id, source_id, connection_string = lookup_es_binding(workspace_id, EVENTSTREAM_NAME, SOURCE_NAME)
    logging.info('Using deployment-bound schema mapping; workspace managed identity is not established')
    settings = amqp_environment(connection_string)
    del connection_string
    pathlib.Path(STATE_FILE).parent.mkdir(parents=True, exist_ok=True)
    from pegelonline_amqp.app import feed
    from pegelonline_core import PegelOnlineAPI
    from pegelonline_amqp_producer_amqp_producer.producer import DeWsvPegelonlineAmqpProducer as producer_class
    failures = []
    def worker():
        try:
            producer = producer_class(host=settings['AMQP_HOST'], port=5671,
                                      address=settings['AMQP_ADDRESS'], use_tls=True,
                                      sas_key_name=settings['AMQP_SAS_KEY_NAME'],
                                      sas_key=settings['AMQP_SAS_KEY'], content_mode='binary',
                                      dataschema_overrides=dataschema_overrides)
            try:
                feed(PegelOnlineAPI(), producer, POLLING_INTERVAL, state_file=STATE_FILE, once=ONCE_MODE, content_type=content_type)
            finally:
                producer.close()
        except BaseException as exc:
            failures.append(exc)
    started = datetime.datetime.now(datetime.timezone.utc).isoformat()
    logging.info('Starting AMQP binary feeder at %s; once=%s; payload=%s', started, ONCE_MODE, content_type)
    thread = threading.Thread(target=worker, daemon=True)
    thread.start()
    thread.join(timeout=RUN_DURATION)
    if failures:
        raise failures[0]
    if errors:
        raise RuntimeError(f'Feeder logged {len(errors)} error(s); inspect last-run.log')
    if thread.is_alive() and ONCE_MODE:
        raise TimeoutError('One-shot feeder did not finish within RUN_DURATION')
    logging.info('Run complete; duration limit reached=%s', thread.is_alive())
except BaseException:
    logging.error('FAILED: %s', traceback.format_exc())
    result = 'FAIL: inspect /lakehouse/default/Files/feeder-state/pegelonline/last-run.log'
finally:
    root_logger.removeHandler(error_handler)
    root_logger.removeHandler(log_handler)
    root_logger.setLevel(old_level)
    log_handler.close()
    if 'settings' in globals():
        settings.clear()
notebookutils.notebook.exit(result)
